In [16]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import gcamreader

def convert_to_mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100‑yr GWP (no climate–carbon feedbacks)
GWP_AR5 = {
    'CO2':      1,      
    'CH4':     28,      
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,      
    'N2O_AGR':265,
    'N2O_AWB':265,
    'HFC125': 3170,     
    'HFC134a':1300,     
    'HFC143a':4800,     
    'HFC23': 12400,     
    'HFC32':   677,     
    'HFC43':  1650,     
    'HFC227ea':3350,    
    'HFC236fa':8060,    
    'SF6':   23500,     
    'C2F6':  11100,     
    'CF4':    6630,     
}

In [18]:
# -----------------------------
# Config
# -----------------------------
PROJECT_PATH = Path("/data/project/tae/gcam-core")
DB_REL_DIR   = "../output"
DB_FILE      = "database_basexdb_korea_2035_v7"
QUERY_FILE   = Path("..") / "output" / "queries" / "Main_queries.xml"
REGION       = "South Korea"
SCENARIOS    = ["High-Ambition-Med", 'High-Ambition-Med-CPO2040']

# save result
OUT_PNG = "./fig/emiss_by_sec.png"

In [19]:
# -----------------------------
# Helpers
# -----------------------------
def convert_to_mt(row: pd.Series) -> float:
    """Convert GCAM emission units to Mt."""
    val, unit = row.get("value"), row.get("Units")
    if unit == "Tg":
        return float(val)
    if unit == "Gg":
        return float(val) * 1e-3
    if unit == "MTC":
        # convert MtC → MtCO2
        return float(val) * (44.009 / 12.011)
    raise ValueError(f"Unknown unit: {unit}")

def connect_db() -> gcamreader.LocalDBConn:
    return gcamreader.LocalDBConn(DB_REL_DIR, DB_FILE)

def run_query(conn, idx: int, scenarios=SCENARIOS, region=REGION) -> pd.DataFrame:
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[idx]
    df = conn.runQuery(q, scenarios=scenarios, regions=[region])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def strip_d_suffix(s: str) -> str:
    # remove trailing _d1.._d10 (downscaler suffix)
    return re.sub(r"_d(?:[1-9]|10)$", "", s)

In [20]:
# -----------------------------
# Load mapping tables
# -----------------------------
dfCO2Map   = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfNonCO2Map= pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])

In [21]:
# -----------------------------
# Data pull
# -----------------------------
conn = connect_db()

Database scenarios: High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-High, High-Ambition-Low, High-Ambition-Med-AI, High-Ambition-Med-CPO2040, Current-Policies-Low, Current-Policies-High, Current-Policies-Med-AI


In [22]:
# CO2 (query 262) & non-CO2 (query 272) — keep your original indices
dfCO2 = run_query(conn, 262)
dfCO2["sector"] = dfCO2["sector"].map(strip_d_suffix)
dfCO2["GHG"] = "CO2"

dfNonCO2 = run_query(conn, 272)
dfNonCO2["sector"] = dfNonCO2["sector"].map(strip_d_suffix)
dfNonCO2 = dfNonCO2[dfNonCO2["GHG"].isin(GWP_AR5.keys())].copy()

In [23]:
# -----------------------------
# Split out special non-CO2 subsectors (fires & waste in urban processes)
# -----------------------------
mask_fires = (dfNonCO2["sector"].eq("UnmanagedLand")) & (dfNonCO2["subsector"].isin(["ForestFire", "GrasslandFires"]))
mask_waste = (dfNonCO2["sector"].eq("urban processes")) & (dfNonCO2["subsector"].isin(["landfills", "wastewater", "waste_incineration"]))

dfNonCO2_1 = dfNonCO2[~(mask_fires | mask_waste)].copy()
dfNonCO2_2 = dfNonCO2[(mask_fires | mask_waste)].copy()

In [24]:
# -----------------------------
# Merge mapping (CO2)
# -----------------------------
cols_map = ["sector", "var1", "var2", "var3", "var4", "var5"]
dfCO2Sec = dfCO2.merge(dfCO2Map[cols_map].drop_duplicates(), on="sector", how="left")

In [25]:
# -----------------------------
# Merge mapping (non-CO2)
#   (1) rows that map only by sector+ghg
#   (2) rows that map by sector+subsector+ghg
# -----------------------------
map1 = dfNonCO2Map[dfNonCO2Map["subsector"].isna()][["sector", "ghg", "var1", "var2", "var3", "var4", "var5"]].drop_duplicates()
map2 = dfNonCO2Map[["sector", "subsector", "ghg", "var1", "var2", "var3", "var4", "var5"]].drop_duplicates()

dfNonCO2_1Sec = dfNonCO2_1.merge(map1, left_on=["sector", "GHG"], right_on=["sector", "ghg"], how="left")
dfNonCO2_2Sec = dfNonCO2_2.merge(map2, left_on=["sector", "subsector", "GHG"], right_on=["sector", "subsector", "ghg"], how="left")
dfNonCO2Sec   = pd.concat([dfNonCO2_1Sec, dfNonCO2_2Sec], ignore_index=True)

In [26]:
# -----------------------------
# Sector categorization
# -----------------------------
def cat_sec_co2(row: pd.Series) -> str:
    sec = row["sector"]
    v2  = str(row.get("var2") or "")
    v5  = str(row.get("var5") or "")
    if sec == "electricity": return "Electricity"
    if sec == "agricultural energy use": return "Agriculture"
    if v2.endswith("Other Capture and Removal"): return "DAC"
    if sec == "cement": return "Industry"
    if sec == "desalinated water": return "Buildings"
    if v5.endswith("Industry"): return "Industry"
    if v5.endswith("Electricity"): return "Electricity"
    if v5.endswith("Residential and Commercial"): return "Buildings"
    if v5.endswith("Transportation"): return "Transportation"
    if sec in ["delivered biomass", "delivered gas", "gas pipeline", "gas processing",
               "refined liquids enduse", "refined liquids industrial", "refining", "wholesale gas"]:
        return "Industry"
    if v5.endswith("AFOFI"): return "Industry"
    return "Others"

def cat_sec_nonco2(row: pd.Series) -> str:
    ghg = row["GHG"]
    v2  = str(row.get("var2") or "")
    v3  = str(row.get("var3") or "")
    v4  = str(row.get("var4") or "")
    sec = row["sector"]

    # gas grouping
    if ghg in ["HFC125", "HFC134a", "HFC143a", "HFC23", "HFC32", "HFC43", "HFC227ea", "HFC236fa", "SF6", "C2F6", "CF4"]:
        return "F-Gases"
    if ghg in ["CH4", "CH4_AWB", "CH4_AGR"]:
        return "Methane"

    # sector logic
    if sec == "agricultural energy use": return "Agriculture"
    if v2.endswith("Waste"): return "Waste"
    if v2.endswith("AFOLU") and (v3.endswith("Agriculture") or v3.endswith("Agricultural Waste Burning")):
        return "Agriculture"
    if v2.endswith("AFOLU"): return "Others"
    if v2.endswith("Industrial Processes") or sec == "industrial processes": return "Industry"
    if v2.endswith("Waste") or sec == "urban processes": return "Others"
    if v4.endswith("Electricity"): return "Electricity"
    if v2.endswith("Industrial Processes") or v4.endswith("Industry"): return "Industry"
    if v4.endswith("Transportation"): return "Transportation"
    if v4.endswith("Residential and Commercial"): return "Buildings"
    if v4.endswith("AFOFI") or v4.endswith("Heat") or v4.endswith("Liquids"): return "Industry"
    return "Others"

dfCO2Sec["sec"]    = dfCO2Sec.apply(cat_sec_co2, axis=1)
dfNonCO2Sec["sec"] = dfNonCO2Sec.apply(cat_sec_nonco2, axis=1)

In [27]:
# -----------------------------
# Compute MtCO2e
# -----------------------------
dfGHGSec = pd.concat([dfCO2Sec, dfNonCO2Sec], ignore_index=True)
dfGHGSec["emiss(MT)"] = dfGHGSec.apply(convert_to_mt, axis=1)
dfGHGSec["gwpAr5"]    = dfGHGSec["GHG"].map(GWP_AR5)
dfGHGSec["MTCO2eq"]   = dfGHGSec["emiss(MT)"] * dfGHGSec["gwpAr5"]

In [28]:
dfGHGSec[(dfGHGSec['sec'] == 'Industry')]['sector'].unique()

array(['ammonia', 'cement', 'chemical energy use', 'chemical feedstocks',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat dac',
       'process heat food processing', 'process heat paper',
       'refined liquids enduse', 'refined liquids industrial', 'refining',
       'waste biomass for paper', 'wholesale gas', 'industrial processes'],
      dtype=object)

In [29]:
dfGHGSec[(~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario                   Year
High-Ambition-Med          1975     54.590644
                           1990    295.171616
                           2005    588.834536
                           2010    694.980605
                           2015    750.755896
                           2020    750.034773
                           2025    689.473277
                           2030    518.111874
                           2035    358.523074
High-Ambition-Med-CPO2040  1975     54.590644
                           1990    295.171616
                           2005    588.834536
                           2010    694.980605
                           2015    750.755896
                           2020    750.034773
                           2025    689.473277
                           2030    532.929923
                           2035    390.617358
Name: MTCO2eq, dtype: float64

In [19]:
(379.803055 - 48.2) / 742.3

0.4467237707126499

In [31]:
358.523074 - 48.2

310.323074

In [32]:
390.617358 - 48.2

342.41735800000004

In [17]:
345.156072 - 47.8

297.356072

In [33]:
dfGHGSecDiff = dfGHGSec[(dfGHGSec['Year'].isin([2020, 2035]) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))) & (dfGHGSec['scenario'].isin(SCENARIOS))].groupby(['scenario', 'Year', 'sec'])['MTCO2eq'].sum().reset_index().pivot(index=['scenario', 'sec'], columns=['Year'], values='MTCO2eq')
dfGHGSecDiff.fillna(0, inplace=True)
dfGHGSecDiff['Diff'] = dfGHGSecDiff[2035] - dfGHGSecDiff[2020]
dfGHGSecDiff

Year                                            2020        2035        Diff
scenario                  sec                                               
High-Ambition-Med         Agriculture       7.547345    6.525990   -1.021355
                          Buildings        55.380619   21.771227  -33.609391
                          DAC               0.000000   -5.915622   -5.915622
                          Electricity     243.273286   46.409610 -196.863676
                          F-Gases          39.590313   17.886313  -21.704000
                          Industry        254.603582  184.707337  -69.896245
                          Methane          26.682134   18.483658   -8.198477
                          Others            0.053019    0.985060    0.932041
                          Transportation  122.014426   67.034175  -54.980250
                          Waste             0.890050    0.635325   -0.254725
High-Ambition-Med-CPO2040 Agriculture       7.547345    6.459572   -1.087773
                          Buildings        55.380619   19.943660  -35.436959
                          DAC               0.000000   -5.915622   -5.915622
                          Electricity     243.273286   93.230896 -150.042390
                          F-Gases          39.590313   17.941674  -21.648639
                          Industry        254.603582  172.088791  -82.514791
                          Methane          26.682134   18.471572   -8.210562
                          Others            0.053019    0.977810    0.924791
                          Transportation  122.014426   66.783680  -55.230745
                          Waste             0.890050    0.635325   -0.254725

In [17]:
# -----------------------------
# Sector reductions (given constants from your notes)
# -----------------------------
emiss_2018_total = 783.8
emiss_2018_net   = 742.3
emiss_2035_ep    = 297.356072

# deltas
power_diff   = 50.45
ind_diff     = 16.02
trn_diff     = 1.92
bld_diff     = 5.20
methane_diff = 1.60
fgas_diff    = -4.05
others_diff  = -0.22

# 2018 sector levels
power_2018   = 278.78
ind_2018     = 269.18
trn_2018     = 98.05
bld_2018     = 48.32
methane_2018 = 37.16
fgas_2018    = 33.80
others_2018  = 18.58

In [18]:
def get_diff(df, scenario: str, sector: str) -> float:
    try:
        return float(df.loc[(scenario, sector), 'Diff'])
    except KeyError:
        return 0.0

# --- Combine Agriculture + Waste + Others ---
def get_combined_others(df, scenario):
    return (
        get_diff(df, scenario, 'Agriculture')
      + get_diff(df, scenario, 'Waste')
      + get_diff(df, scenario, 'Others')
      - 1.3
    )

In [19]:
CP, EP = SCENARIOS

# Waterfall stepping bases & contributions
power_base = emiss_2018_net
power_ep   = get_diff(dfGHGSecDiff, EP, 'Electricity') - power_diff
power_cp   = get_diff(dfGHGSecDiff, CP, 'Electricity') - power_diff

ind_base = power_base + power_ep
ind_ep   = get_diff(dfGHGSecDiff, EP, 'Industry') - ind_diff
ind_cp   = get_diff(dfGHGSecDiff, CP, 'Industry') - ind_diff

trn_base = ind_base + ind_ep
trn_ep   = get_diff(dfGHGSecDiff, EP, 'Transportation') - trn_diff
trn_cp   = get_diff(dfGHGSecDiff, CP, 'Transportation') - trn_diff

bld_base = trn_base + trn_ep
bld_ep   = get_diff(dfGHGSecDiff, EP, 'Buildings') - bld_diff
bld_cp   = get_diff(dfGHGSecDiff, CP, 'Buildings') - bld_diff

methane_base = bld_base + bld_ep
methane_ep   = get_diff(dfGHGSecDiff, EP, 'Methane') - methane_diff
methane_cp   = get_diff(dfGHGSecDiff, CP, 'Methane') - methane_diff

fgas_base = methane_base + methane_ep
fgas_ep   = get_diff(dfGHGSecDiff, EP, 'F-Gases') - fgas_diff
fgas_cp   = get_diff(dfGHGSecDiff, CP, 'F-Gases')  - fgas_diff

dac_base = fgas_base + fgas_ep
dac_ep   = -5.915622
dac_cp   =  0.0

lulucf_base = dac_base + dac_ep
lulucf_ep   = -6.8
lulucf_cp   =  0.0

others_base = lulucf_base + lulucf_ep
others_ep   = get_combined_others(dfGHGSecDiff, EP) - others_diff   # model difference 포함
others_cp   = get_combined_others(dfGHGSecDiff, EP) - others_diff

# % reduction by 2018 sector levels (Enhanced)
rr_power   = -(power_ep   / power_2018)   * 100
rr_ind     = -(ind_ep     / ind_2018)     * 100
rr_trn     = -(trn_ep     / trn_2018)     * 100
rr_bld     = -(bld_ep     / bld_2018)     * 100
rr_methane = -(methane_ep / methane_2018) * 100
rr_fgas    = -(fgas_ep    / fgas_2018)    * 100
rr_others  = -(others_ep  / others_2018)  * 100

# Total reduction vs 2018 TOTAL
rd_ttl = emiss_2035_ep - emiss_2018_net
rr_ttl = -rd_ttl / emiss_2018_net * 100

In [20]:
data = pd.DataFrame({
    "category": [
        "2018 (TOTAL)", "2018 (NET)",
        "Power", "Industry", "Transport", "Buildings", "Methane", "F-Gases", "DAC", "LULUCF", "Other", 
        "2035"
    ],
})

fig = go.Figure()

# Start and end bars
fig.add_trace(go.Waterfall(
    name="Enhanced Ambition",
    orientation="v",
    measure=["absolute"] * 2 + ["relative"] * 9 + ["total"],
    x=data["category"],
    y=[emiss_2018_total, emiss_2018_net, power_ep, ind_ep, trn_ep, bld_ep, methane_ep, fgas_ep, dac_ep, lulucf_ep, others_ep, emiss_2035_ep],
    base=0,
    connector={"visible": False},
    decreasing={"marker": {"color": "#1f77b4"}},   # enhanced ambition
    increasing={"marker": {"color": "#FF6692"}},   # current policies
    totals={"marker": {"color": "lightgray"}},
    showlegend=False
))


# Add Current Policy overlays just for Coal categories
fig.add_trace(go.Bar(
    name="Current Policy",
    x=["Power", "Industry", "Transport", "Buildings", "Methane", "F-Gases", "DAC", "LULUCF", "Other"],
    y=[power_cp, ind_cp, trn_cp, bld_cp, methane_cp, fgas_cp, dac_cp, lulucf_cp, others_cp],  # smaller reductions
    base=[power_base, ind_base, trn_base, bld_base, methane_base, fgas_base, dac_base, lulucf_base, others_base],  # position on top of previous waterfall step
    marker_color="#AEC7E8",
    showlegend=False
))

fig.update_layout(
    width=1050,
    height=500,
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    # title="<b>Emissions Reductions from Each Sector Compared to 2018 Levels</b>",
    # title_font_size=21,
    # title_x=0.5,

)

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["2018 (TOTAL)", "2018 (NET)", "2035"],
    y=[emiss_2018_net / 2, emiss_2018_net / 2, emiss_2035_ep / 2],
    mode="text",
    text=[f"<b>{emiss_2018_total:.1f}<br>TOTAL</b>", f"<b>{emiss_2018_net:.1f}<br>NET</b>", f"<b>{emiss_2035_ep:.1f}<br>NET</b>"],
    textposition="middle center",
    showlegend=False
))

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["Power", "Industry", "Transport", "Buildings", "Methane", "F-Gases", "DAC", "LULUCF", 'Other'],
    #-190.5, -73.8, -28.9, -15.9, -4.5, -4.9, -3.0, -9
    y=[x+40 for x in [power_base, ind_base, trn_base, bld_base, methane_base, fgas_base]] + [y + 10 for y in [dac_base, lulucf_base]] + [others_base + 40],
    mode="text",
    text=[f"<b>{power_ep:.1f}<br>(△{rr_power:.1f}%)</b>", f"<b>{ind_ep:.1f}<br>(△{rr_ind:.1f}%)</b>", f"<b>{trn_ep:.1f}<br>(△{rr_trn:.1f}%)</b>", 
          f"<b>{bld_ep:.1f}<br>(△{rr_bld:.1f}%)</b>", f"<b>{methane_ep:.1f}<br>(△{rr_methane:.1f}%)</b>", f"<b>{fgas_ep:.1f}<br>(△{rr_fgas:.1f}%)</b>",
          f"<b>{dac_ep:.1f}</b>", f"<b>{lulucf_ep:.1f}</b>", f"<b>{others_ep:.1f}<br>(△{rr_others:.1f}%)</b>"],
    textposition="middle center",
    showlegend=False
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Current Policies",
    marker=dict(color="#AEC7E8"),
    showlegend=True,
    hoverinfo="skip"
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="High Ambition",
    marker=dict(color="#1f77b4"),
    showlegend=True,
    hoverinfo="skip"
))


fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=15)
    )
)

fig.update_layout(
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title="Emission (MtCO2e)", title_font_size=18,
        tickvals=list(range(0, 801, 100)),
    )
)

fig.add_annotation(
    x=11, y=emiss_2035_ep + 5,        # 7 is the index of "2035" in the x-category list
    ax=11, ay=emiss_2018_net + 30,
    xref="x", yref="y",
    axref="x", ayref="y",
    text=f"<b>{rd_ttl:.1f}<br>(△{rr_ttl:.1f}%)</b>",
    showarrow=True,
    arrowhead=3,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1f77b4",
    font=dict(size=12, color="black"),
    align="center"
)

fig.add_annotation(
    x=2 + 0.2, y=ind_base + 5,        # 7 is the index of "2035" in the x-category list
    ax=2 + 0.2, ay=emiss_2018_net,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>Enhanced<br>Ambition</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#00A08B",
    font=dict(size=10, color="#00A08B"),
    align="center"
)

fig.add_annotation(
    x=2 -0.2, y=emiss_2018_net + power_cp + 5,        # 7 is the index of "2035" in the x-category list
    ax=2 - 0.2, ay=emiss_2018_net,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>CoalOut</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1616A7",
    font=dict(size=10, color="#1616A7"),
    align="center"
)


fig.update_layout(
    xaxis=dict(
        tickvals=list(range(len(data['category']))),
        ticktext=[f"<b>{cat}</b>" for cat in data['category']],
        # tickfont=dict(size=15)
    )
)

fig.add_annotation(
    text="<b>Current<br>Policies</b>",
    # xref="paper", yref="paper",
    x=2-0.6, y=670,
    showarrow=False,
    font=dict(size=10, color="#1616A7"),
    align="left"
)

fig.add_annotation(
    text="<b>Enhanced<br>Ambition</b>",
    # xref="paper", yref="paper",
    x=2+0.65, y=670,
    showarrow=False,
    font=dict(size=10, color="#00A08B"),
    align="left"
)

# fig.update_layout(
#     legend=dict(
#         orientation='h',
#         yanchor='bottom',
#         y=-0.35,  # move legend lower (more negative = lower)
#         xanchor='center',
#         x=0.5,
#         bgcolor='rgba(0,0,0,0)',
#         borderwidth=0,
#         font=dict(size=15)
#     )
# )

fig.update_layout(
    margin=dict(l=40, r=40, t=40, b=120),  # increased bottom margin
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.23,  # move legend up a bit so it's visible
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=15)
    )
)

# DAC explanatory note below legend
# fig.add_annotation(
#     text="*DAC (Direct Air Capture) represents negative emissions from atmospheric CO₂ removal.",
#     xref="paper", yref="paper",
#     x=0, y=-0.3,  # just below legend
#     showarrow=False,
#     font=dict(size=11, color="black"),
#     align="left"
# )

# fig.add_annotation(
#     text="**Other includes residual emissions from sectors not covered under CH4 or F-gases, such as agriculture, waste, and fugitive emissions.",
#     xref="paper", yref="paper",
#     x=0, y=-0.35,  # just below legend
#     showarrow=False,
#     font=dict(size=11, color="black"),
#     align="left"
# )

fig.update_layout(
    xaxis=dict(
        tickvals=list(range(len(data['category']))),
        ticktext=[
            "<b>2018<br>(Total)</b>",
            "<b>2018<br>(Net)</b>",
            "<b>Power</b>", "<b>Industry</b>", "<b>Transport</b>",
            "<b>Buildings</b>", "<b>Methane</b>", "<b>F-Gases</b>",
            "<b>DAC</b>", "<b>LULUCF</b>", "<b>Other</b>",
            "<b>2035</b>"
        ],
        tickangle=0,
        tickfont=dict(size=12)
    )
)

# pio.write_image(fig, "./fig/emiss_by_sec.png", width=1050, height=500, scale=2)
pio.write_image(fig, "./fig/emiss_by_sec.jpg", width=1050, height=500, scale=3)
pio.write_image(fig, "./fig/emiss_by_sec.svg", width=1050, height=500, scale=3)
fig

In [19]:
def cat_ind_sec(sector):
    if sector in ['iron and steel']:
        return 'Iron and Steel'
    elif sector in ['ammonia', 'chemical energy use', 'chemical feedstocks',]:
        return 'Chemical'
    elif sector in ['cement', 'process heat cement']:
        return 'Cement'
    else:
        return 'Other Industry'

In [20]:
dfGHGSecInd = dfGHGSec[(dfGHGSec['sec'] == 'Industry')].copy()
dfGHGSecInd['ind_sec'] = dfGHGSecInd['sector'].apply(cat_ind_sec)

In [21]:
dfGHGSecIndDiff = dfGHGSecInd[(dfGHGSecInd['Year'].isin([2020, 2035])) & (dfGHGSecInd['scenario'].isin(SCENARIOS))].groupby(['scenario', 'Year', 'ind_sec'])['MTCO2eq'].sum().reset_index().pivot(index=['scenario', 'ind_sec'], columns=['Year'], values='MTCO2eq')
dfGHGSecIndDiff.fillna(0, inplace=True)
dfGHGSecIndDiff['Diff'] = dfGHGSecIndDiff[2035] - dfGHGSecIndDiff[2020]
dfGHGSecIndDiff

Year                                       2020        2035       Diff
scenario              ind_sec                                         
Current-Policies-Med  Cement          32.738486   29.688899  -3.049586
                      Chemical         6.208311    6.702753   0.494442
                      Iron and Steel  88.890435   72.465475 -16.424960
                      Other Industry  91.982072   96.661405   4.679333
Enhanced-Ambition-Med Cement          33.820055   24.454769  -9.365286
                      Chemical         6.525703   -6.058387 -12.584091
                      Iron and Steel  90.305123   30.149027 -60.156096
                      Other Industry  92.798392  100.268201   7.469809

In [22]:
emiss_2018 = 269.98

steel_diff = 7.06
chem_diff = 2.35
cement_diff = 2.50
other_diff = 0.07

steel_2018 = 111.48
chem_2018 = 54.84
cement_2018 = 42.63
other_2018 = 93.92

In [23]:
steel_base = emiss_2018
steel_ep = get_diff(dfGHGSecIndDiff, EP, 'Iron and Steel') - steel_diff 
steel_ep

-67.21609620513607

In [24]:
steel_base = emiss_2018
steel_ep = get_diff(dfGHGSecIndDiff, EP, 'Iron and Steel') - steel_diff 
steel_cp = get_diff(dfGHGSecIndDiff, CP, 'Iron and Steel') - steel_diff 
chem_base = emiss_2018 + steel_ep
chem_ep = get_diff(dfGHGSecIndDiff, EP, 'Chemical') - chem_diff
chem_cp = get_diff(dfGHGSecIndDiff, CP, 'Chemical') - chem_diff
cement_base = chem_base + chem_ep
cement_ep = get_diff(dfGHGSecIndDiff, EP, 'Cement') - cement_diff
cement_cp = get_diff(dfGHGSecIndDiff, CP, 'Cement') - cement_diff
other_base = cement_base + cement_ep
other_ep = 7.46 - other_diff -4.05
other_cp = 5.23 - other_diff -4.05

In [25]:

emiss_2035_ep = emiss_2018 - 90.7

rr_steel =  - (steel_ep / steel_2018) * 100
rr_chem = - (chem_ep / chem_2018) * 100
rr_cement = - (cement_ep / cement_2018) * 100
rr_other = - (other_ep / other_2018) * 100

In [26]:
rd_ttl = emiss_2035_ep - emiss_2018
rr_ttl = -rd_ttl / emiss_2018 * 100

In [32]:
data = pd.DataFrame({
    "category": [
        "2018",
        "Iron & Steel", "Chemical", "Cement", "Other Industry",
        "2035"
    ],
})

fig = go.Figure()

# Start and end bars
fig.add_trace(go.Waterfall(
    name="High Ambition",
    orientation="v",
    measure=["absolute"] * 1 + ["relative"] * 4 + ["total"],
    x=data["category"],
    y=[emiss_2018, steel_ep, chem_ep, cement_ep, other_ep, emiss_2035_ep],
    base=0,
    connector={"visible": False},
    decreasing={"marker": {"color": "#1f77b4"}},   # enhanced ambition
    increasing={"marker": {"color": "#FF6692"}},   # current policies
    totals={"marker": {"color": "lightgray"}},
    showlegend=False
))

# Add Current Policy overlays just for Coal categories
fig.add_trace(go.Bar(
    name="Current Policy",
    x=["Iron & Steel", "Chemical", "Cement", "Other Industry"],
    y=[steel_cp, chem_cp, cement_cp, other_cp],  # smaller reductions
    base=[steel_base, chem_base, cement_base, other_base],  # position on top of previous waterfall step
    marker_color="#AEC7E8",
    showlegend=False
))

fig.update_layout(
    width=600,
    height=500,
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',


)

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["2018", "2035"],
    y=[emiss_2018 / 2, emiss_2035_ep / 2],
    mode="text",
    text=[f"<b>{emiss_2018:.1f}</b>", f"<b>{emiss_2035_ep:.1f}</b>"],
    textposition="middle center",
    showlegend=False
))

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["Iron & Steel", "Chemical", "Cement", "Other Industry"],
    #-190.5, -73.8, -28.9, -15.9, -4.5, -4.9, -3.0, -9
    y=[x+20 for x in [steel_base, chem_base, cement_base, other_base]],
    mode="text",
    text=[f"<b>{steel_ep:.1f}<br>(△{rr_steel:.1f}%)</b>", f"<b>{chem_ep:.1f}<br>(△{rr_chem:.1f}%)</b>", f"<b>{cement_ep:.1f}<br>(△{rr_cement:.1f}%)</b>", 
          f"<b>{other_ep:.1f}<br>(△{rr_other:.1f}%)</b>"],
    textposition="middle center",
    showlegend=False
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Current Policies",
    marker=dict(color="#AEC7E8"),
    showlegend=True,
    hoverinfo="skip"
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="High Ambition",
    marker=dict(color="#1f77b4"),
    showlegend=True,
    hoverinfo="skip"
))


fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=15)
    )
)

fig.update_layout(
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title="Emission (MtCO2e)", title_font_size=18,
        tickvals=list(range(0,351, 50)),
    )
)

fig.add_annotation(
    x=5, y=emiss_2035_ep + 5,        # 7 is the index of "2035" in the x-category list
    ax=5, ay=emiss_2018 + 30,
    xref="x", yref="y",
    axref="x", ayref="y",
    text=f"<b>{rd_ttl:.1f}<br>(△{rr_ttl:.1f}%)</b>",
    showarrow=True,
    arrowhead=3,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1f77b4",
    font=dict(size=12, color="black"),
    align="center"
)

fig.add_annotation(
    x=1 + 0.2, y=chem_base,        # 7 is the index of "2035" in the x-category list
    ax=1 + 0.2, ay=emiss_2018 + 3,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>Enhanced<br>Ambition</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=2,
    arrowsize=1,
    arrowcolor="#00A08B",
    font=dict(size=10, color="#00A08B"),
    align="center"
)

fig.add_annotation(
    x=1 -0.2, y=emiss_2018 + steel_cp - 2,        # 7 is the index of "2035" in the x-category list
    ax=1 - 0.2, ay=emiss_2018 + 3,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>CoalOut</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=2,
    arrowsize=1,
    arrowcolor="#1616A7",
    font=dict(size=10, color="#1616A7"),
    align="center"
)


fig.update_layout(
    xaxis=dict(
        tickvals=list(range(len(data['category']))),
        ticktext=[f"<b>{cat}</b>" for cat in data['category']],
        # tickfont=dict(size=15)
    )
)

fig.add_annotation(
    text="<b>Current<br>Policies</b>",
    # xref="paper", yref="paper",
    x=1-0.5, y=259,
    showarrow=False,
    font=dict(size=8, color="#1616A7"),
    align="left"
)

fig.add_annotation(
    text="<b>High<br>Ambition</b>",
    # xref="paper", yref="paper",
    x=1+0.6, y=259,
    showarrow=False,
    font=dict(size=8, color="#00A08B"),
    align="left"
)
pio.write_image(fig, "./fig/emiss_by_industry_sector.png", width=600, height=500, scale=2)

fig